[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheGhoul21/QC-notebooks/blob/main/QC-Intro/01_phases_and_interference.ipynb)

# Fasi e Interferenza — Come Calcola Davvero il Quantum Computing

**Notebook 1** della serie *QC-Intro: Il Quantum Computing attraverso l'Interferenza*


In [ ]:
# ── Colab setup (skip if running locally) ──
try:
    import google.colab
    !pip install -q ipywidgets matplotlib numpy qiskit
    !wget -qO qc_intro_utils.py https://raw.githubusercontent.com/TheGhoul21/QC-notebooks/main/QC-Intro/qc_intro_utils.py
    print("✓ Colab setup complete")
except ImportError:
    pass  # running locally

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

from qc_intro_utils import (
    apply_style, draw_bloch_sphere, draw_amplitude_bars,
    draw_interference_arrows, draw_circuit_amplitude_evolution,
    draw_phase_pattern, phase_to_color,
    PRIMARY_COLOR, ACCENT_COLOR, GHOST_COLOR, EDGE_CMAP
)

apply_style()
%matplotlib inline

---

## 1. La fase — la variabile nascosta

Consideriamo due stati:

$$|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \qquad |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

Entrambi hanno **probabilità identiche**: $P(0) = P(1) = 1/2$. Nessuna misura diretta nella base computazionale può distinguerli.

Ma applicando il gate di Hadamard $H$:

$$H|+\rangle = |0\rangle, \qquad H|-\rangle = |1\rangle$$

Il risultato è **completamente diverso**! La **fase relativa** tra le componenti — invisibile alla misura diretta — controlla tutto quando un gate agisce sullo stato.

### Forward: $|+\rangle$ e $|-\rangle$ prima e dopo $H$

Visualizziamo le ampiezze dei due stati fianco a fianco. Stesso modulo, fasi diverse. Poi applichiamo $H$ e vediamo come il risultato diverge completamente.

In [ ]:
# Hadamard matrix
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)

# |+> and |-> states
plus  = np.array([1, 1], dtype=complex) / np.sqrt(2)
minus = np.array([1, -1], dtype=complex) / np.sqrt(2)

# After Hadamard
H_plus  = H @ plus
H_minus = H @ minus

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

draw_amplitude_bars(plus,  ax=axes[0, 0], title=r'$|+\rangle$ — before $H$', show_phase_colorbar=False)
draw_amplitude_bars(minus, ax=axes[0, 1], title=r'$|-\rangle$ — before $H$', show_phase_colorbar=False)
draw_amplitude_bars(H_plus,  ax=axes[1, 0], title=r'$H|+\rangle = |0\rangle$', show_phase_colorbar=False)
draw_amplitude_bars(H_minus, ax=axes[1, 1], title=r'$H|-\rangle = |1\rangle$', show_phase_colorbar=False)

plt.tight_layout()
plt.show()

### Backward: "Voglio che applicando $H$ ottenga $|1\rangle$ con certezza. Da che stato devo partire?"

Poiché $H = H^\dagger$ (l'Hadamard è il proprio inverso), basta applicare $H$ all'output desiderato:

$$H|1\rangle = |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

Quindi devo partire da $|-\rangle$: la fase relativa di $\pi$ tra le due componenti è **necessaria** per ottenere interferenza distruttiva su $|0\rangle$ e costruttiva su $|1\rangle$.

In [ ]:
# Verify: H applied to |1> gives |->
ket1 = np.array([0, 1], dtype=complex)
result = H @ ket1
print(f"H|1⟩ = {result[0]:.4f}|0⟩ + {result[1]:.4f}|1⟩")
print(f"This is |−⟩: coefficients are {1/np.sqrt(2):.4f} and {-1/np.sqrt(2):.4f}")

### Explore: fase relativa e probabilità dopo $H$

Varia la fase relativa $\varphi$ nello stato $(|0\rangle + e^{i\varphi}|1\rangle)/\sqrt{2}$. Osserva come $P(0)$ e $P(1)$ dopo $H$ cambiano con continuità — la fase controlla il destino dello stato.

In [ ]:
def explore_phase(phi=0.0):
    """Show state before and after H as relative phase varies."""
    state = np.array([1, np.exp(1j * phi)], dtype=complex) / np.sqrt(2)
    after_H = H @ state
    probs = np.abs(after_H)**2

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    draw_amplitude_bars(state, ax=axes[0], title=r'Before $H$', show_phase_colorbar=False)
    draw_amplitude_bars(after_H, ax=axes[1], title=r'After $H$', show_phase_colorbar=False)

    # Probability bars
    axes[2].bar([0, 1], probs, color=[PRIMARY_COLOR, ACCENT_COLOR])
    axes[2].set_xticks([0, 1])
    axes[2].set_xticklabels([r'$P(0)$', r'$P(1)$'])
    axes[2].set_ylim(0, 1.1)
    axes[2].set_title(f'$P(0)={probs[0]:.2f}$, $P(1)={probs[1]:.2f}$')
    for i, p in enumerate(probs):
        axes[2].text(i, p + 0.03, f'{p:.3f}', ha='center', fontsize=10)

    plt.tight_layout()
    plt.show()

interact(explore_phase,
         phi=FloatSlider(value=0, min=0, max=2*np.pi, step=0.1,
                         description='φ (phase)', readout_format='.2f'));

---

## 2. Interferenza di due cammini

L'analogia fondamentale è con l'**esperimento della doppia fenditura**. Quando due ampiezze complesse $a$ e $b$ si combinano, il risultato dipende dalla loro **fase relativa**:

- **Interferenza costruttiva** (fasi allineate): $|a + b| > \max(|a|, |b|)$ — le frecce puntano nella stessa direzione
- **Interferenza distruttiva** (fasi opposte): $|a + b| \approx 0$ — le frecce si cancellano

Nel piano complesso, sommare ampiezze è sommare vettori **punta-coda**. La lunghezza del vettore risultante determina la probabilità.

### Forward: tre casi di interferenza

Due ampiezze con uguale modulo ($|a| = |b| = 0.5$) e fasi diverse: allineate (costruttiva), opposte (distruttiva), e un caso intermedio.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Constructive: both amplitudes point right
draw_interference_arrows([0.5 + 0j, 0.5 + 0j], ax=axes[0],
                         title='Costruttiva (Δφ = 0)')

# Destructive: opposite directions
draw_interference_arrows([0.5 + 0j, -0.5 + 0j], ax=axes[1],
                         title='Distruttiva (Δφ = π)')

# Intermediate: 90° phase difference
draw_interference_arrows([0.5 + 0j, 0.5j], ax=axes[2],
                         title='Intermedia (Δφ = π/2)')

plt.tight_layout()
plt.show()

### Backward: "Voglio cancellazione perfetta. Che angolo relativo serve?"

Per avere $|a + b| = 0$ con $|a| = |b|$, serve che $b = -a$, cioè una differenza di fase $\Delta\varphi = \pi$. Le frecce devono essere **antiparallele**: il secondo vettore torna esattamente all'origine.

In [ ]:
# Perfect cancellation demonstration
a = 0.5 * np.exp(1j * 0.3)  # arbitrary phase
b = 0.5 * np.exp(1j * (0.3 + np.pi))  # phase shifted by pi

fig, ax = plt.subplots(figsize=(6, 6))
draw_interference_arrows([a, b], ax=ax,
                         title=f'Cancellazione perfetta: |a+b| = {abs(a+b):.6f}')
plt.show()

### Explore: interferenza con due fasi variabili

Usa i due slider per controllare la fase di ciascuna ampiezza (modulo fisso a 0.5). Osserva come il modulo della somma varia: scoprirai che la cancellazione avviene esattamente quando $\Delta\varphi = \pi$.

In [ ]:
def explore_interference(phase_1=0.0, phase_2=0.0):
    """Interactive interference of two complex amplitudes."""
    a = 0.5 * np.exp(1j * phase_1)
    b = 0.5 * np.exp(1j * phase_2)
    total = a + b
    delta = (phase_2 - phase_1) % (2 * np.pi)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    draw_interference_arrows([a, b], ax=axes[0],
                             title=f'Δφ = {delta:.2f} rad')

    # Bar showing magnitude of sum
    axes[1].bar(['|a+b|'], [abs(total)], color=ACCENT_COLOR, width=0.4)
    axes[1].set_ylim(0, 1.1)
    axes[1].set_title(f'|a+b| = {abs(total):.3f}')
    axes[1].axhline(1.0, color='gray', ls='--', lw=0.5, label='max (constructive)')
    axes[1].axhline(0.0, color='gray', ls=':', lw=0.5)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(explore_interference,
         phase_1=FloatSlider(value=0, min=0, max=2*np.pi, step=0.1,
                             description='φ₁', readout_format='.2f'),
         phase_2=FloatSlider(value=0, min=0, max=2*np.pi, step=0.1,
                             description='φ₂', readout_format='.2f'));

---

## 3. Hadamard come beam-splitter

La matrice di Hadamard:

$$H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

agisce come un **beam-splitter quantistico**:

- $H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ — superposizione con fasi positive
- $H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$ — superposizione con una fase negativa su $|1\rangle$

Proprietà fondamentale: $H \cdot H = I$. Applicare $H$ due volte riporta allo stato originale — i "fasci" si ricombinano perfettamente. Questo è l'analogo quantistico di un **interferometro di Mach-Zehnder**.

Al secondo $H$, le ampiezze si sommano:
- Per $|0\rangle$: $\frac{1}{\sqrt{2}}(\frac{1}{\sqrt{2}} + \frac{1}{\sqrt{2}}) = 1$ (costruttiva)
- Per $|1\rangle$: $\frac{1}{\sqrt{2}}(\frac{1}{\sqrt{2}} - \frac{1}{\sqrt{2}}) = 0$ (distruttiva)

### Forward: $|0\rangle \to H \to H \to |0\rangle$

Mostriamo l'evoluzione delle ampiezze passo per passo: lo stato iniziale, dopo il primo $H$, dopo il secondo $H$.

In [ ]:
# |0> -> H -> H -> |0>
ket0 = np.array([1, 0], dtype=complex)
step1 = H @ ket0   # |+>
step2 = H @ step1   # back to |0>

stages = [
    (r'$|0\rangle$', ket0),
    (r'After $H$: $|+\rangle$', step1),
    (r'After $HH$: $|0\rangle$', step2),
]

fig, axes = draw_circuit_amplitude_evolution(stages)
fig.suptitle(r'Interferometro: $H \cdot H = I$', fontsize=13, y=1.05)
plt.show()

### Backward: "Quale stato produce $(a, b)$ arbitrario dopo $H$?"

Siccome $H = H^\dagger$, basta applicare $H$ all'output desiderato:

$$|\psi_{\text{input}}\rangle = H |\psi_{\text{output}}\rangle$$

Questa è la potenza dell'invertibilità unitaria: possiamo sempre risalire dallo stato finale a quello iniziale.

In [ ]:
# Example: what input gives output (0.6, 0.8) after H?
desired_output = np.array([0.6, 0.8], dtype=complex)
required_input = H @ desired_output  # H^{-1} = H

# Verify
check = H @ required_input
print(f"Desired output: {desired_output}")
print(f"Required input: {required_input}")
print(f"Verification H @ input: {check}")

### Explore: $H$ come interferometro

Varia $\theta$ e $\varphi$ sullo stato di input e osserva tre pannelli: ampiezze di input, dopo $H$, dopo $HH$. Noterai che $HH$ riporta **sempre** allo stato iniziale.

In [ ]:
def explore_hadamard(theta=0.0, phi=0.0):
    """Explore H as interferometer: input -> H -> HH."""
    state_in = np.array([np.cos(theta / 2),
                         np.exp(1j * phi) * np.sin(theta / 2)], dtype=complex)
    after_H = H @ state_in
    after_HH = H @ after_H

    stages = [
        ('Input', state_in),
        ('After H', after_H),
        ('After HH', after_HH),
    ]
    fig, axes = draw_circuit_amplitude_evolution(stages)
    plt.show()

interact(explore_hadamard,
         theta=FloatSlider(value=0, min=0, max=np.pi, step=0.1,
                           description='θ', readout_format='.2f'),
         phi=FloatSlider(value=0, min=0, max=2*np.pi, step=0.1,
                         description='φ', readout_format='.2f'));

---

## 4. Multi-qubit: interferenza su $2^n$ cammini

Con $n$ qubit, la superposizione ha $2^n$ ampiezze. $H^{\otimes n}$ crea superposizione uniforme. Se poi applichiamo **phase kicks condizionali** e un altro $H^{\otimes n}$, le fasi interferiscono su **tutti** i $2^n$ stati simultaneamente.

Questo **NON** è "calcolare $2^n$ cose contemporaneamente". È preparare $2^n$ ampiezze in modo che l'interferenza **concentri la probabilità** dove vogliamo.

**Esempio con 2 qubit**: $|00\rangle \xrightarrow{H^{\otimes 2}}$ uniforme $\xrightarrow{\text{CZ}}$ phase kick su $|11\rangle$ $\xrightarrow{H^{\otimes 2}}$ interferenza.

### Forward: interferenza su 2 qubit con CZ

Partiamo da $|00\rangle$, applichiamo $H^{\otimes 2}$ per creare superposizione uniforme, poi CZ (che flippa la fase di $|11\rangle$), e infine $H^{\otimes 2}$ di nuovo.

In [ ]:
# Build circuit step by step, capturing state at each stage
qc = QuantumCircuit(2)
sv_init = Statevector.from_instruction(qc)

qc.h([0, 1])
sv_after_h = Statevector.from_instruction(qc)

qc.cz(0, 1)
sv_after_cz = Statevector.from_instruction(qc)

qc.h([0, 1])
sv_final = Statevector.from_instruction(qc)

stages = [
    (r'$|00\rangle$', sv_init.data),
    (r'After $H^{\otimes 2}$', sv_after_h.data),
    (r'After CZ', sv_after_cz.data),
    (r'After $H^{\otimes 2}$ again', sv_final.data),
]

fig, axes = draw_circuit_amplitude_evolution(stages)
fig.suptitle('Interferenza multi-qubit: H⊗2 → CZ → H⊗2', fontsize=13, y=1.05)
plt.show()

print("Final probabilities:")
for i, amp in enumerate(sv_final.data):
    print(f"  |{i:02b}⟩: P = {abs(amp)**2:.4f}")

### Backward: "Ho questo pattern di output. Che fasi devo mettere prima dell'ultimo $H^{\otimes n}$?"

Siccome $(H^{\otimes n})^{-1} = H^{\otimes n}$, basta applicare $H^{\otimes n}$ all'output desiderato per trovare le fasi necessarie:

$$|\psi_{\text{prima dell'ultimo } H^{\otimes n}}\rangle = H^{\otimes n} |\psi_{\text{output}}\rangle$$

In [ ]:
# Backward: I want all probability on |10> (state index 2)
desired = np.array([0, 0, 1, 0], dtype=complex)  # |10>

# H tensor H matrix
H2 = np.kron(H, H)
required_phases = H2 @ desired

print("To get output |10⟩, the state before the final H⊗2 must be:")
for i, amp in enumerate(required_phases):
    print(f"  |{i:02b}⟩: {amp.real:+.4f} {amp.imag:+.4f}j")

# Verify
check = H2 @ required_phases
print(f"\nVerification — H⊗2 applied: {np.round(check, 4)}")

### Explore: pattern di fasi su 2 qubit

4 slider controllano la fase di ciascun stato base ($|00\rangle$, $|01\rangle$, $|10\rangle$, $|11\rangle$) dopo la superposizione uniforme. L'ultimo $H^{\otimes 2}$ produce interferenza. Scopri quale pattern di fasi concentra tutta la probabilità su un unico stato!

In [ ]:
def explore_multiqubit_phases(phi_00=0.0, phi_01=0.0, phi_10=0.0, phi_11=0.0):
    """Apply phase kicks to uniform superposition, then H⊗2."""
    # Uniform superposition
    uniform = np.ones(4, dtype=complex) / 2.0

    # Apply phase kicks
    phases = np.array([phi_00, phi_01, phi_10, phi_11])
    after_phase = uniform * np.exp(1j * phases)

    # Apply H⊗2
    H2 = np.kron(H, H)
    final = H2 @ after_phase
    probs = np.abs(final)**2

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    draw_phase_pattern(after_phase, ax=axes[0], title='After phase kicks')
    draw_amplitude_bars(final, ax=axes[1], title=r'After $H^{\otimes 2}$', show_phase_colorbar=False)

    # Probability distribution
    labels = [f'|{i:02b}⟩' for i in range(4)]
    axes[2].bar(range(4), probs, color=ACCENT_COLOR)
    axes[2].set_xticks(range(4))
    axes[2].set_xticklabels(labels)
    axes[2].set_ylim(0, 1.1)
    axes[2].set_ylabel('Probability')
    axes[2].set_title('Output probability')
    for i, p in enumerate(probs):
        if p > 0.01:
            axes[2].text(i, p + 0.03, f'{p:.2f}', ha='center', fontsize=9)

    plt.tight_layout()
    plt.show()

interact(explore_multiqubit_phases,
         phi_00=FloatSlider(value=0, min=0, max=2*np.pi, step=0.1,
                            description='φ(|00⟩)', readout_format='.2f'),
         phi_01=FloatSlider(value=0, min=0, max=2*np.pi, step=0.1,
                            description='φ(|01⟩)', readout_format='.2f'),
         phi_10=FloatSlider(value=0, min=0, max=2*np.pi, step=0.1,
                            description='φ(|10⟩)', readout_format='.2f'),
         phi_11=FloatSlider(value=0, min=0, max=2*np.pi, step=0.1,
                            description='φ(|11⟩)', readout_format='.2f'));

---

## 5. Phase kickback — come funziona davvero l'oracolo

Un oracolo $U_f$ agisce come:

$$U_f |x\rangle |y\rangle = |x\rangle |y \oplus f(x)\rangle$$

Se l'ancilla è preparata nello stato $|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$, succede qualcosa di magico:

$$U_f |x\rangle |-\rangle = (-1)^{f(x)} |x\rangle |-\rangle$$

**Dimostrazione**:
- Se $f(x) = 0$: $|x\rangle \frac{|0 \oplus 0\rangle - |1 \oplus 0\rangle}{\sqrt{2}} = |x\rangle |-\rangle$
- Se $f(x) = 1$: $|x\rangle \frac{|0 \oplus 1\rangle - |1 \oplus 1\rangle}{\sqrt{2}} = |x\rangle \frac{|1\rangle - |0\rangle}{\sqrt{2}} = -|x\rangle |-\rangle$

La fase **"rimbalza"** (kicks back) sul registro di controllo. L'ancilla rimane inalterata! Questo trucco converte un **oracolo bit-flip** in un **oracolo di fase**.

### Forward: phase kickback passo per passo

Consideriamo 1 qubit + 1 ancilla. L'oracolo è un CNOT (implementa $f(x) = x$). Mostriamo lo stato del sistema completo prima e dopo l'oracolo.

In [ ]:
# Phase kickback with f(x) = x (CNOT oracle)
# Qiskit ordering: qubit 0 = ancilla, qubit 1 = control

# Step 1: prepare |+>|-> (control in |+>, ancilla in |->)
qc_pk = QuantumCircuit(2)
qc_pk.x(0)       # ancilla to |1>
qc_pk.h([0, 1])  # ancilla to |->, control to |+>
sv_before = Statevector.from_instruction(qc_pk)

# Step 2: Apply oracle (CNOT: control=1, target=0)
qc_pk.cx(1, 0)
sv_after = Statevector.from_instruction(qc_pk)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

draw_amplitude_bars(sv_before.data, ax=axes[0],
                    title=r'Before oracle: $|+\rangle|-\rangle$',
                    show_phase_colorbar=False)
draw_amplitude_bars(sv_after.data, ax=axes[1],
                    title=r'After oracle (CNOT)',
                    show_phase_colorbar=False)

plt.tight_layout()
plt.show()

# Verify: control qubit got phase-kicked
print("Before oracle amplitudes:")
for i, amp in enumerate(sv_before.data):
    if abs(amp) > 1e-10:
        print(f"  |{i:02b}⟩: {amp.real:+.4f} {amp.imag:+.4f}j")
print("\nAfter oracle amplitudes:")
for i, amp in enumerate(sv_after.data):
    if abs(amp) > 1e-10:
        print(f"  |{i:02b}⟩: {amp.real:+.4f} {amp.imag:+.4f}j")
print("\nThe control qubit (q1) went from |+⟩ to |−⟩ — the phase kicked back!")

### Backward: "Voglio che l'oracolo marchi $|x\rangle$ con fase $-1$ senza disturbare il target. Che stato deve avere l'ancilla?"

Serve $|-\rangle$! Qualsiasi altro stato dell'ancilla non produce un kickback pulito — l'ancilla si entangle con il controllo e il kickback è "sporco".

In [ ]:
# Show that only |-> gives clean kickback
for ancilla_name, ancilla_state in [('|0⟩', [1, 0]), ('|+⟩', [1/np.sqrt(2), 1/np.sqrt(2)]),
                                     ('|−⟩', [1/np.sqrt(2), -1/np.sqrt(2)])]:
    control = np.array([1, 1], dtype=complex) / np.sqrt(2)  # |+>
    anc = np.array(ancilla_state, dtype=complex)
    full_state = np.kron(control, anc)

    # CNOT matrix (control=qubit0=MSB, target=qubit1=LSB)
    CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
    after = CNOT @ full_state

    # Check if separable (ancilla unchanged)
    print(f"Ancilla = {ancilla_name}:")
    print(f"  Before: {np.round(full_state, 3)}")
    print(f"  After:  {np.round(after, 3)}")
    # Check factorization
    if abs(after[0]*after[3] - after[1]*after[2]) < 1e-10:
        print(f"  → Separable (clean kickback)")
    else:
        print(f"  → Entangled (dirty kickback)")
    print()

### Explore: lo stato dell'ancilla e la qualità del kickback

Varia l'angolo $\alpha$ dell'ancilla: $|\text{anc}\rangle = \cos\alpha|0\rangle + \sin\alpha|1\rangle$. Per $\alpha = -\pi/4$ si ottiene $|-\rangle$. Solo in quel caso il kickback è pulito (nessun entanglement con l'ancilla).

In [ ]:
def explore_kickback(alpha=0.0):
    """Explore how ancilla state affects phase kickback quality."""
    control = np.array([1, 1], dtype=complex) / np.sqrt(2)  # |+>
    ancilla = np.array([np.cos(alpha), np.sin(alpha)], dtype=complex)

    full_before = np.kron(control, ancilla)

    # CNOT: control=q0(MSB), target=q1(LSB)
    CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
    full_after = CNOT @ full_before

    # Measure entanglement via concurrence proxy
    det = abs(full_after[0]*full_after[3] - full_after[1]*full_after[2])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    draw_amplitude_bars(full_before, ax=axes[0],
                        title='Before oracle', show_phase_colorbar=False)
    draw_amplitude_bars(full_after, ax=axes[1],
                        title='After oracle (CNOT)', show_phase_colorbar=False)

    # Entanglement indicator
    color = 'green' if det < 0.01 else 'red'
    axes[2].bar(['Entanglement'], [det], color=color, width=0.4)
    axes[2].set_ylim(0, 0.6)
    axes[2].set_title(f'Entanglement: {det:.3f}\n({"CLEAN" if det < 0.01 else "DIRTY"} kickback)')

    fig.suptitle(f'α = {alpha:.2f} rad, ancilla ≈ {np.cos(alpha):.2f}|0⟩ + {np.sin(alpha):.2f}|1⟩',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

interact(explore_kickback,
         alpha=FloatSlider(value=0, min=-np.pi/2, max=np.pi/2, step=0.05,
                           description='α', readout_format='.2f'));

---

## 6. L'oracolo come marchiatore di fasi

Ora che conosciamo il meccanismo (phase kickback), possiamo capire come un oracolo **marchia** certi stati con un flip di fase:

1. **Prima dell'interferenza**: tutte le ampiezze uguali ($1/\sqrt{N}$)
2. **Dopo l'oracolo**: alcune ampiezze hanno fase $-1$
3. **Dopo l'interferenza** ($H^{\otimes n}$): gli stati marcati vengono **amplificati** (costruttiva), quelli non marcati vengono **soppressi** (distruttiva)

Nota: un singolo round di mark+interfere non amplifica completamente — per quello serve iterare (Grover). Ma già un round mostra l'effetto dell'interferenza.

### Forward: marchiare $|5\rangle$ tra 8 stati

3 qubit, $2^3 = 8$ stati. Superposizione uniforme → marchiamo $|5\rangle$ con $-1$ → applichiamo $H^{\otimes 3}$.

In [ ]:
n_qubits = 3
N = 2**n_qubits

# Step 1: Uniform superposition
uniform = np.ones(N, dtype=complex) / np.sqrt(N)

# Step 2: Mark |5> with phase -1
marked = uniform.copy()
marked[5] *= -1

# Step 3: Apply H⊗3
H3 = np.array([[1]])
for _ in range(n_qubits):
    H3 = np.kron(H3, H)
after_interference = H3 @ marked

stages = [
    ('Uniform superposition', uniform),
    (r'After marking $|5\rangle$', marked),
    (r'After $H^{\otimes 3}$', after_interference),
]

fig, axes = draw_circuit_amplitude_evolution(stages)
fig.suptitle('Oracle marking + interference (3 qubits)', fontsize=13, y=1.05)
plt.show()

print("Final probabilities:")
probs = np.abs(after_interference)**2
for i in range(N):
    marker = " ← marked" if i == 5 else ""
    print(f"  |{i}⟩: P = {probs[i]:.4f}{marker}")

### Backward: "Voglio amplificare lo stato $|5\rangle$ tra 8 possibili. Che oracolo mi serve?"

Un oracolo che assegna $-1$ a $|5\rangle$ e $+1$ a tutto il resto:

$$O|x\rangle = \begin{cases} -|x\rangle & \text{se } x = 5 \\ +|x\rangle & \text{altrimenti} \end{cases}$$

In forma matriciale, è $I - 2|5\rangle\langle 5|$ — l'identità con un singolo segno flippato.

In [ ]:
# Oracle matrix for marking |5>
oracle = np.eye(N, dtype=complex)
oracle[5, 5] = -1

print("Oracle diagonal (non-identity elements):")
for i in range(N):
    if oracle[i, i] != 1:
        print(f"  O[{i},{i}] = {oracle[i,i]:.0f}  ← marks |{i}⟩")

### Explore: scegli quali stati marchiare

Per 3 qubit (8 stati), seleziona quali stati marchiare con $-1$. Osserva le ampiezze prima/dopo la marchiatura/dopo l'interferenza. Prova a marchiare combinazioni diverse per vedere come cambia il pattern di output.

In [ ]:
def explore_oracle_marking(s0=False, s1=False, s2=False, s3=False,
                           s4=False, s5=True, s6=False, s7=False):
    """Mark selected states and show interference effect."""
    marked_states = [s0, s1, s2, s3, s4, s5, s6, s7]
    n_q = 3
    N_states = 2**n_q

    # Uniform superposition
    uniform_sv = np.ones(N_states, dtype=complex) / np.sqrt(N_states)

    # Apply marks
    marked_sv = uniform_sv.copy()
    for i, do_mark in enumerate(marked_states):
        if do_mark:
            marked_sv[i] *= -1

    # H⊗3
    Hn = np.array([[1]])
    for _ in range(n_q):
        Hn = np.kron(Hn, H)
    final_sv = Hn @ marked_sv

    stages = [
        ('Uniform', uniform_sv),
        ('After marking', marked_sv),
        (r'After $H^{\otimes 3}$', final_sv),
    ]

    fig, axes = draw_circuit_amplitude_evolution(stages)

    marked_labels = [f'|{i}⟩' for i, m in enumerate(marked_states) if m]
    fig.suptitle(f'Marked: {marked_labels if marked_labels else "none"}',
                 fontsize=11, y=1.05)
    plt.show()

interact(explore_oracle_marking,
         s0=widgets.Checkbox(value=False, description='|0⟩'),
         s1=widgets.Checkbox(value=False, description='|1⟩'),
         s2=widgets.Checkbox(value=False, description='|2⟩'),
         s3=widgets.Checkbox(value=False, description='|3⟩'),
         s4=widgets.Checkbox(value=False, description='|4⟩'),
         s5=widgets.Checkbox(value=True, description='|5⟩'),
         s6=widgets.Checkbox(value=False, description='|6⟩'),
         s7=widgets.Checkbox(value=False, description='|7⟩'));

---

## 7. Il paradigma — prepare → mark → interfere → measure

Riassumiamo il paradigma del quantum computing:

1. **Prepare**: Crea superposizione uniforme ($H^{\otimes n}$)
2. **Mark**: L'oracolo applica phase kicks via kickback
3. **Interfere**: Un altro layer di $H^{\otimes n}$ (o interferenza più sofisticata)
4. **Measure**: L'interferenza costruttiva ha concentrato la probabilità sulla risposta

Questo **NON** è "provare tutto contemporaneamente". È **ingegnerizzare le fasi** affinché la fisica faccia il filtraggio.

Il "parallelismo quantistico" è un'**interferenza ingegnerizzata**: si preparano le condizioni affinché le risposte sbagliate si cancellino e quella giusta emerga.

### Forward: la pipeline completa su 2 qubit

Mostriamo tutte e 4 le fasi. L'oracolo marca $|3\rangle = |11\rangle$. Con prepare + mark + interfere, la risposta emerge.

In [ ]:
# Complete pipeline: 2 qubits, mark |3> = |11>
n_q = 2
N_s = 2**n_q

# 1. Prepare
prepared = np.ones(N_s, dtype=complex) / np.sqrt(N_s)

# 2. Mark |3>
marked_pipeline = prepared.copy()
marked_pipeline[3] *= -1

# 3. Interfere
H2_mat = np.kron(H, H)
interfered = H2_mat @ marked_pipeline

stages = [
    ('1. Prepare\n(H⊗2)', prepared),
    ('2. Mark\n(oracle on |11⟩)', marked_pipeline),
    ('3. Interfere\n(H⊗2)', interfered),
]

fig, axes = draw_circuit_amplitude_evolution(stages)
fig.suptitle('Pipeline completa: prepare → mark → interfere', fontsize=13, y=1.05)
plt.show()

probs_pipeline = np.abs(interfered)**2
print("4. Measure — probabilità:")
for i in range(N_s):
    bar = '█' * int(probs_pipeline[i] * 40)
    print(f"  |{i:02b}⟩: {probs_pipeline[i]:.4f} {bar}")

### Backward: "Cosa succede se rimuovo una fase della pipeline?"

Senza *prepare*: nessuna superposizione su cui operare. Senza *mark*: nessuna informazione codificata nelle fasi. Senza *interfere*: le fasi sono invisibili alla misura. **Ogni fase è indispensabile.**

In [ ]:
# Show what happens when each stage is removed
target_state = 3  # |11>

def run_pipeline(do_prepare=True, do_mark=True, do_interfere=True):
    """Run pipeline with optional stages."""
    n_q_loc = 2
    N_loc = 2**n_q_loc
    H2_loc = np.kron(H, H)

    # Start from |00>
    state = np.zeros(N_loc, dtype=complex)
    state[0] = 1.0

    if do_prepare:
        state = H2_loc @ state  # uniform superposition
    if do_mark:
        state[target_state] *= -1  # phase kick
    if do_interfere:
        state = H2_loc @ state  # interference

    return state

configs = [
    ('Full pipeline', True, True, True),
    ('No prepare', False, True, True),
    ('No mark', True, False, True),
    ('No interfere', True, True, False),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (label, prep, mark, interf) in zip(axes, configs):
    final_state = run_pipeline(prep, mark, interf)
    probs_cfg = np.abs(final_state)**2
    colors = [ACCENT_COLOR if i == target_state else PRIMARY_COLOR for i in range(4)]
    ax.bar(range(4), probs_cfg, color=colors)
    ax.set_xticks(range(4))
    ax.set_xticklabels([f'|{i:02b}⟩' for i in range(4)], fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_title(label, fontsize=10)

axes[0].set_ylabel('Probability')
fig.suptitle(f'Target: |{target_state:02b}⟩ — ogni fase è essenziale', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

### Explore: pipeline interattiva

Attiva/disattiva ciascuna fase della pipeline e osserva come cambia la distribuzione di probabilità. Con tutte le fasi attive, la risposta emerge. Rimuovi una qualsiasi fase e la risposta scompare.

In [ ]:
def explore_pipeline(prepare=True, mark=True, interfere=True):
    """Interactive pipeline with toggleable stages."""
    n_q_loc = 2
    N_loc = 2**n_q_loc
    H2_loc = np.kron(H, H)
    target = 3  # |11>

    state = np.zeros(N_loc, dtype=complex)
    state[0] = 1.0

    stage_list = [('Initial |00⟩', state.copy())]

    if prepare:
        state = H2_loc @ state
        stage_list.append(('After prepare (H⊗2)', state.copy()))

    if mark:
        state[target] *= -1
        stage_list.append(('After mark (−1 on |11⟩)', state.copy()))

    if interfere:
        state = H2_loc @ state
        stage_list.append(('After interfere (H⊗2)', state.copy()))

    fig, axes = draw_circuit_amplitude_evolution(stage_list)

    status = []
    if prepare: status.append('PREPARE ✓')
    else: status.append('prepare ✗')
    if mark: status.append('MARK ✓')
    else: status.append('mark ✗')
    if interfere: status.append('INTERFERE ✓')
    else: status.append('interfere ✗')
    fig.suptitle(' → '.join(status), fontsize=11, y=1.05)
    plt.show()

interact(explore_pipeline,
         prepare=widgets.Checkbox(value=True, description='Prepare (H⊗2)'),
         mark=widgets.Checkbox(value=True, description='Mark (oracle)'),
         interfere=widgets.Checkbox(value=True, description='Interfere (H⊗2)'));

---

## Riepilogo

| Concetto | Intuizione chiave |
|----------|------------------|
| **Fase relativa** | Invisibile alla misura diretta, ma controlla tutto quando i gate agiscono |
| **Interferenza** | Somma di ampiezze complesse: costruttiva (fasi allineate) o distruttiva (fasi opposte) |
| **Hadamard** | Beam-splitter quantistico: crea e ricombina superposizioni |
| **Multi-qubit** | $2^n$ ampiezze che interferiscono simultaneamente — non "calcolo parallelo" |
| **Phase kickback** | Converte un oracolo bit-flip in un oracolo di fase tramite ancilla in $|-\rangle$ |
| **Oracle marking** | Marchia stati con $-1$; l'interferenza successiva amplifica i marcati |
| **Il paradigma** | **Prepare → Mark → Interfere → Measure**: interferenza ingegnerizzata |

Il quantum computing non è magia né calcolo parallelo. È **fisica dell'interferenza**, sfruttata con precisione chirurgica per far emergere la risposta giusta da un mare di possibilità.